<a href="https://colab.research.google.com/github/wjohn564/CS6271-2025-6---Final-Project/blob/main/20245424_EA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
<img src="https://www.ul.ie/themes/custom/ul/logo.jpg" />
</div>

#**MSc in Artificial Intelligence and Machine Learning**
##CS6271 - Evolutionary Algorithms and Humanoid Robotics 2025
### Kaggle Competition


Module Leader: Conor Ryan

Developer: John Walsh

Predict whether a person will earn more than 50k or less. This is a modified version of the adult dataset

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Dataset

**Class:**

output: 1, 0.


**Listing of features:**

- 'age'
- 'workclass'
- 'fnlwgt'
- 'education'
- 'education-num'
- 'marital-status'
- 'occupation'
- 'relationship'
- 'race'
- 'sex'
- 'capital-gain'
- 'capital-loss'
- 'hours-per-week'
- 'native-country'

In [ ]:
# Suppressing Warnings:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
## mount your Google drive
# 1) run this cell
# 2) sign in
# 3) verify your drive is mounted

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Remember you will need grape to run Grammatical Evolution

In [ ]:
import os
# Get the library from our BDS research Group
# copy the path from your drive
PATH = '/content/drive/MyDrive/grape/'

# check if 'grape' already exists
if os.path.exists(PATH):
    print('grape directory already exists')
else:
    %cd /content/drive/MyDrive/
    !git clone https://github.com/bdsul/grape.git
    print('Cloning grape in your Drive')

# change directory to 'grape'
%cd /content/drive/MyDrive/grape/

/content/drive/MyDrive
Cloning into 'grape'...
remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 356 (delta 81), reused 82 (delta 73), pack-reused 256 (from 1)
Receiving objects: 100% (356/356), 5.90 MiB | 7.80 MiB/s, done.
Resolving deltas: 100% (189/189), done.
Cloning grape in your Drive
/content/drive/MyDrive/grape


## Load the training dataset, visualise some stats and create the target variable

In [ ]:
# Load the Training Set
train_file = '/content/train.csv'
df_train = pd.read_csv(train_file)

# Show the first 5 rows
df_train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,37,Private,193106,Bachelors,13,Never-married,Sales,Not-in-family,White,Female,0,0,30,United-States,0
1,56,Self-emp-inc,216636,12th,8,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,1651,40,United-States,0
2,53,Private,126977,HS-grad,9,Separated,Craft-repair,Not-in-family,White,Male,0,0,35,United-States,0
3,72,Private,205343,11th,7,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
4,46,State-gov,106705,Masters,14,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,38,United-States,0


In [ ]:
# Visualise some helpful details
df_train.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income
count,39073.000000,3.907300e+04,39073.000000,39073.000000,39073.000000,39073.000000,39073.000000
mean,38.643488,1.899922e+05,10.069844,1038.040540,86.807949,40.476877,0.239270
std,13.685634,1.054768e+05,2.574387,7204.953114,401.276773,12.401251,0.426643
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,1.177670e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.786150e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,48.000000,2.383290e+05,12.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [ ]:
# More stats to view
df_train.describe(include='object')

,workclass,education,marital-status,occupation,relationship,race,sex,native-country
count,38304,39073,39073,38302,39073,39073,39073,38851
unique,9,16,7,15,6,5,2,42
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States
freq,27141,12582,17951,4915,15822,33414,26170,35059


In [ ]:
X_train = df_train.copy()
# warning: cannot drop it more than once
X_train.drop(['income'], axis=1, inplace=True)

Y_train = df_train['income'].to_numpy()

## Test set: **No labels**

In [ ]:
# you can load the dataset from you drive like this.
# train_file = 'datasets/test.csv'
# df_test = pd.read_csv(PATH+train_file)
# df_test.head()


# I am loading it after uploading directly

train_file = 'test.csv'
df_test = pd.read_csv(train_file)
df_test.head()

In [ ]:
df_test.describe()

In [ ]:
df_test.describe(include='object')

In [ ]:
X_test = df_test.copy()

You will need to prepare both training and test datasets before working with a Machine Learning method.

Consider you need to use some encoding method with categorical data.

You are free to use any other pre-processing ideas.

## GRAPE

<div>
<img src="https://drive.google.com/uc?export=view&id=1hw43Oi3lGTCkspQ0ged2bZB8q2EpcPhz" width="150"/>
</div>

GRammatical Algorithms in Python for Evolution (GRAPE)

In [ ]:
!pip install deap

import grape
import algorithms

from os import path
from deap import creator, base, tools
import random
import csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 3.2 MB/s eta 0:00:00


You can import functions to be used with your grammar from [functions.py](https://github.com/UL-BDS/grape/blob/main/functions.py) on GRAPE repository and / or you can define your own functions.

In [ ]:
from functions import add, sub, mul, pdiv, psqrt, plog, and_, or_, not_, less_than_or_equal, greater_than_or_equal

'heartDisease.bnf' is a grammar used for another problem just to check if everything is working well.

Write your own grammar in a text file and save it in your Drive account.

Put the whole address on GRAMMAR_FILE and print to check it.

In [ ]:
#GRAMMAR_FILE = '/content/drive/MyDrive/data/example.bnf' #put the whole address of your own grammar and remove the # in the beginning of this line
GRAMMAR_FILE = 'heartDisease.bnf' #remove this line when you are using your own grammar

#f = open(GRAMMAR_FILE, "r") #remove the # in the beginning of this line when you are using your own grammar
f = open( GRAMMAR_FILE, "r") #remove this line when you are using your own grammar
print(f.read())
f.close()

Run the following cell to put your grammar on the class Grammar.

In [ ]:
BNF_GRAMMAR = grape.Grammar(GRAMMAR_FILE)#change this line when you are using your own grammar

The fitness function here is the percentage of outputs wrongly predicted.

You can write your own fitness function if you prefer.

In [ ]:
def fitness_eval(individual, points):
    """
    Fitness Function
    """

    x = points[0]
    Y = points[1]

    if individual.invalid == True:
        return np.NaN,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.NaN,
    assert np.isrealobj(pred)

    compare = np.equal(Y,pred)
    fitness = 1 - np.mean(compare)

    return fitness,

To use properly the fitness function above with GRAPE, the features must be in the lines, and the samples must be in the columns, so if your data is not like that, you need to transpose the matrix.

Take a look at the print. If you run this cell two times, the matrix will be transposed again and will not work properly.

In [ ]:
X_train = np.transpose(X_train)
X_test = np.transpose(X_test)

print('Training (X,Y):\t', X_train.shape, Y_train.shape)
print('Test (X):\t', X_test.shape)

Set the Grammatical Evolution parameters.

Make sure you set a random seed just in case we need to re-run your experiments.

In [ ]:
POPULATION_SIZE =
MAX_GENERATIONS =
P_CROSSOVER =
P_MUTATION =
ELITE_SIZE =
HALLOFFAME_SIZE =

TOURNAMENT_SIZE =
RANDOM_SEED =
random.seed(RANDOM_SEED)

CODON_CONSUMPTION = 'lazy'
GENOME_REPRESENTATION = 'list'
MAX_GENOME_LENGTH = None

MAX_INIT_TREE_DEPTH =
MIN_INIT_TREE_DEPTH =
MAX_TREE_DEPTH =
MAX_WRAPS = 0
CODON_SIZE = 255

REPORT_ITEMS = ['gen', 'invalid', 'avg', 'std', 'min', 'max',
                'best_ind_length', 'avg_length',
                'best_ind_nodes', 'avg_nodes',
                'best_ind_depth', 'avg_depth',
                'avg_used_codons', 'best_ind_used_codons',
                'structural_diversity', 'fitness_diversity',
                'selection_time', 'generation_time']

Create a toolbox

In [ ]:
toolbox = base.Toolbox()

# define a single objective, minimising fitness strategy:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

creator.create('Individual', grape.Individual, fitness=creator.FitnessMin)

toolbox.register("populationCreator", grape.sensible_initialisation, creator.Individual)

toolbox.register("evaluate", fitness_eval)

# Tournament selection:
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

# Single-point crossover:
toolbox.register("mate", grape.crossover_onepoint)

# Flip-int mutation:
toolbox.register("mutate", grape.mutation_int_flip_per_codon)

In [ ]:
# create initial population (generation 0):
population = toolbox.populationCreator(pop_size=POPULATION_SIZE,
                                           bnf_grammar=BNF_GRAMMAR,
                                           min_init_depth=MIN_INIT_TREE_DEPTH,
                                           max_init_depth=MAX_INIT_TREE_DEPTH,
                                           codon_size=CODON_SIZE,
                                           codon_consumption=CODON_CONSUMPTION,
                                           genome_representation=GENOME_REPRESENTATION
                                            )

# define the hall-of-fame object:
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

# prepare the statistics object:
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.nanmean)
stats.register("std", np.nanstd)
stats.register("min", np.nanmin)
stats.register("max", np.nanmax)

Run GE

In [ ]:
population, logbook = algorithms.ge_eaSimpleWithElitism(population, toolbox, cxpb=P_CROSSOVER, mutpb=P_MUTATION,
                                              ngen=MAX_GENERATIONS, elite_size=ELITE_SIZE,
                                              bnf_grammar=BNF_GRAMMAR,
                                              codon_size=CODON_SIZE,
                                              max_tree_depth=MAX_TREE_DEPTH,
                                              max_genome_length=MAX_GENOME_LENGTH,
                                              points_train=[X_train, Y_train],
                                              codon_consumption=CODON_CONSUMPTION,
                                              report_items=REPORT_ITEMS,
                                              genome_representation=GENOME_REPRESENTATION,
                                              stats=stats, halloffame=hof, verbose=False)

show the best individual as an expression

In [ ]:
# Best individual
import textwrap
best = hof.items[0].phenotype
print("Best individual: \n","\n".join(textwrap.wrap(best,80)))
print("\nTraining Fitness: ", hof.items[0].fitness.values[0])

Define a function to predict values, without comparing to expected outputs.

In [ ]:
def predict(individual, X):
    x = X

    if individual.invalid == True:
        return np.NaN,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.NaN,
    assert np.isrealobj(pred)

    return pred

Predict the classes of the test set.

Make sure you print here in the notebook you will submit to Brightspace the same predictions you used in your best submission to the Kaggle competition.

In [ ]:
y_pred = predict(hof.items[0], X_test)
print("Predicted classes of the test set: ", y_pred)

Write a code to create a .csv with the following format:
1. First column is the index (from 0 to 95803);
2. Second column is named `output` and contains the predictions (only 0's or 1's) you  got in the previous cell with y_pred.

Example:

    index,output

    0,0

    1,0

    2,1

    ...

    95803,0


Submit it to the competition and check your score there.